# 08 — Dot Products, Norms & Distance Metrics
## How to Measure Similarity and Size

> **Prerequisites:** Notebooks 01–07  
> **Difficulty:** ⭐⭐☆☆☆ Beginner-Intermediate  
> **Running example:** Comparing house similarity, regularization

---

## What are we learning?

If vectors are arrows, then:
- **Norms** answer: *"How long is this arrow?"*
- **Dot products** answer: *"How aligned are these two arrows?"*
- **Distances** answer: *"How far apart are these two points?"*

These appear constantly in ML:
- **Loss functions** use norms to measure error
- **Regularization** uses norms to penalize large weights  
- **KNN** uses distance to find nearest neighbors
- **Attention mechanisms** use dot products for similarity scores
- **Cosine similarity** measures document similarity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## 1. Dot Product — Measuring Alignment

### Algebraic definition (how to compute it)
> **a · b = a₁×b₁ + a₂×b₂ + ... + aₙ×bₙ = Σᵢ aᵢbᵢ**

Multiply each pair of matching elements and add them all up.

### Geometric definition (what it means)
> **a · b = ‖a‖ × ‖b‖ × cos(θ)**

Where θ is the angle between the two vectors.

### What the result tells you
| Result | Meaning | Angle |
|---|---|---|
| Large positive | Very similar direction | θ close to 0° |
| Zero | Perpendicular (no alignment) | θ = 90° |
| Large negative | Opposite directions | θ close to 180° |

### In ML
- Every neuron output is: `z = w · x + b` (a dot product!)
- **Attention**: query · key scores = how much to attend to each token
- **Cosine similarity**: normalized dot product = direction similarity

In [ ]:
# ─────────────────────────────────────────────────────────
# DOT PRODUCT
# ─────────────────────────────────────────────────────────

print("=== Dot product: three ways to compute ===")
print()

a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])

# Method 1: manual loop (show what's happening)
dot_manual = sum(a[i]*b[i] for i in range(len(a)))
print(f"Manual: a·b = {a[0]}×{b[0]} + {a[1]}×{b[1]} + {a[2]}×{b[2]} = {dot_manual}")

# Method 2: np.dot
dot_np = np.dot(a, b)
print(f"np.dot(a, b) = {dot_np}")

# Method 3: @ operator (also works for vectors)
dot_at = a @ b
print(f"a @ b = {dot_at}")

print()
print("=== Geometric interpretation: angle between vectors ===")
norm_a = np.linalg.norm(a)
norm_b = np.linalg.norm(b)
cos_theta = np.dot(a, b) / (norm_a * norm_b)
theta = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
print(f"||a|| = {norm_a:.4f}")
print(f"||b|| = {norm_b:.4f}")
print(f"cos(θ) = {cos_theta:.4f}")
print(f"θ = {theta:.2f}°")

print()
print("=== Different angles ===")
pairs = [
    (np.array([1.,0.]), np.array([1.,0.]),  "same direction"),
    (np.array([1.,0.]), np.array([0.,1.]),  "perpendicular"),
    (np.array([1.,0.]), np.array([-1.,0.]), "opposite"),
    (np.array([1.,0.]), np.array([1.,1.])/np.sqrt(2), "45° apart"),
]
for va, vb, desc in pairs:
    dot = np.dot(va, vb)
    theta = np.degrees(np.arccos(np.clip(dot, -1, 1)))
    print(f"  {desc:15s}: dot={dot:.4f}, θ={theta:.0f}°")

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUAL: dot product = projection × length
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cases = [
    (np.array([3.,1.]), np.array([2.,3.]), "a·b = large positive"),
    (np.array([3.,0.]), np.array([0.,2.]), "a·b = 0 (perpendicular)"),
    (np.array([3.,0.]), np.array([-2.,1.]), "a·b = negative"),
]

for ax, (va, vb, title) in zip(axes, cases):
    dot = np.dot(va, vb)
    ax.annotate('', xy=va, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.5))
    ax.annotate('', xy=vb, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='orange', lw=2.5))
    ax.text(va[0]+0.1, va[1], 'a', color='steelblue', fontsize=12, fontweight='bold')
    ax.text(vb[0]+0.1, vb[1], 'b', color='orange', fontsize=12, fontweight='bold')
    ax.axhline(0,'k',lw=0.5); ax.axvline(0,'k',lw=0.5)
    ax.set_xlim(-3,4); ax.set_ylim(-2,4)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(f'{title}\na·b = {dot:.1f}', fontsize=10)

plt.suptitle('Dot Product = How Aligned Two Vectors Are', fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. Vector Norms — Measuring Length

### What is a norm?
A **norm** ‖v‖ measures the "size" or "length" of a vector. Different norms measure size differently.

### The p-norm family
> **‖v‖ₚ = (|v₁|ᵖ + |v₂|ᵖ + ... + |vₙ|ᵖ)^(1/p)**

| Norm | p | Computation | Shape of "unit ball" | ML use |
|---|---|---|---|---|
| **L0** | 0 | Count non-zeros | — | Measure sparsity |
| **L1** (Manhattan) | 1 | Sum of \|values\| | Diamond | LASSO regularization |
| **L2** (Euclidean) | 2 | √(sum of squares) | Circle/sphere | Ridge regularization |
| **L∞** (Max) | ∞ | Maximum \|value\| | Square | Minimax problems |

### L1 vs L2 — When to use which?
- **L1** promotes **sparsity**: many weights become exactly 0 (feature selection!)
- **L2** promotes **small values**: all weights shrink proportionally (stability)
- **L1 is robust to outliers**: outliers don't get squared
- **L2 penalizes outliers more**: because it squares them

In [ ]:
# ─────────────────────────────────────────────────────────
# NORMS: L0, L1, L2, L-infinity
# ─────────────────────────────────────────────────────────

v = np.array([3., -4., 0., 2., -1.])
print("=== Different norms of the same vector ===")
print(f"v = {v}")
print()
print(f"L0 norm = {np.count_nonzero(v)}   ← just counts non-zeros")
print(f"         ({np.count_nonzero(v)} non-zero elements out of {len(v)})")
print()
print(f"L1 norm = {np.linalg.norm(v, 1):.1f}   ← |3| + |-4| + |0| + |2| + |-1|")
print(f"         = {int(abs(v[0]))}+{int(abs(v[1]))}+{int(abs(v[2]))}+{int(abs(v[3]))}+{int(abs(v[4]))} = {int(np.linalg.norm(v,1))}")
print()
print(f"L2 norm = {np.linalg.norm(v, 2):.4f}   ← √(3²+4²+0²+2²+1²) = √30")
print(f"         = √({int(v[0]**2)}+{int(v[1]**2)}+{int(v[2]**2)}+{int(v[3]**2)}+{int(v[4]**2)}) = √{int(sum(v**2))} ≈ {np.sqrt(sum(v**2)):.4f}")
print()
print(f"L∞ norm = {np.linalg.norm(v, np.inf):.1f}   ← max(|3|,|-4|,|0|,|2|,|-1|) = 4")

print()
print("=== L1 is robust to outliers ===")
v_normal  = np.array([1., 1., 1., 1., 1.])
v_outlier = np.array([1., 1., 1., 1., 10.])  # one big outlier
print(f"Normal  : L1={np.linalg.norm(v_normal, 1):.0f},  L2={np.linalg.norm(v_normal, 2):.3f}")
print(f"Outlier : L1={np.linalg.norm(v_outlier, 1):.0f}, L2={np.linalg.norm(v_outlier, 2):.3f}")
print(f"L1 grew by: {np.linalg.norm(v_outlier,1)/np.linalg.norm(v_normal,1):.1f}x")
print(f"L2 grew by: {np.linalg.norm(v_outlier,2)/np.linalg.norm(v_normal,2):.1f}x  ← much more sensitive!")

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUAL: unit balls for different norms
# All vectors of 'length 1' look different for different norms
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
theta = np.linspace(0, 2*np.pi, 1000)

norm_settings = [
    (0.5, 'L0.5 (p<1)', 'star shape'),
    (1,   'L1 (Manhattan)', 'diamond'),
    (2,   'L2 (Euclidean)', 'circle'),
    (np.inf, 'L∞ (Max)', 'square'),
]

for ax, (p, name, shape_desc) in zip(axes, norm_settings):
    pts = []
    for t in theta:
        direction = np.array([np.cos(t), np.sin(t)])
        if p == np.inf:
            norm_val = max(abs(direction))
        elif p < 1:  # p=0.5: special case
            norm_val = np.sum(np.abs(direction)**p)**(1/p)
        else:
            norm_val = np.sum(np.abs(direction)**p)**(1/p)
        pts.append(direction / norm_val)
    pts = np.array(pts)
    
    ax.fill(pts[:,0], pts[:,1], alpha=0.3, color='steelblue')
    ax.plot(pts[:,0], pts[:,1], 'steelblue', lw=2)
    ax.axhline(0,'k',lw=0.5); ax.axvline(0,'k',lw=0.5)
    ax.set_xlim(-1.8,1.8); ax.set_ylim(-1.8,1.8)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.set_title(f'{name}\nUnit ball = {shape_desc}', fontsize=9)

plt.suptitle('Unit Ball = All Vectors With Norm = 1 (different norms → different shapes)', fontsize=11)
plt.tight_layout()
plt.show()

---
## 3. Frobenius Norm — The Matrix Size

### What is it?
**Flatten the matrix to a vector, then take L2 norm.**
> **‖A‖_F = √(Σᵢⱼ Aᵢⱼ²)**

It's the "overall size" of the matrix.

### ML uses
- **Weight regularization**: penalize ‖W‖_F² to prevent overfitting
- **Measuring approximation error**: how different is A from A_k?
- **Gradient monitoring**: is ‖∇W‖_F too large? (exploding gradient)

In [ ]:
# ─────────────────────────────────────────────────────────
# FROBENIUS NORM
# ─────────────────────────────────────────────────────────

A = np.array([[1., 2., 3.],
              [4., 5., 6.]])

frob = np.linalg.norm(A, 'fro')         # numpy built-in
frob_manual = np.sqrt((A**2).sum())      # manual: sum all squares, take sqrt
print(f"A:\n{A}")
print(f"‖A‖_F = {frob:.4f}")
print(f"Manual: √(1²+2²+3²+4²+5²+6²) = √{int((A**2).sum())} = {frob_manual:.4f}")

print()
print("=== ML: L2 regularization uses Frobenius norm ===")
np.random.seed(0)
W = np.random.randn(50, 30)  # a weight matrix in a neural network
lambda_reg = 0.01             # regularization strength

l2_penalty = lambda_reg * np.linalg.norm(W, 'fro')**2
print(f"Weight matrix W: {W.shape}")
print(f"‖W‖_F = {np.linalg.norm(W, 'fro'):.4f}")
print(f"L2 penalty (λ‖W‖_F²) = {l2_penalty:.4f}")
print(f"This gets added to the loss: total_loss = task_loss + {lambda_reg} × ‖W‖²")
print(f"Effect: gradients push W toward zero → prevents overfitting")

---
## 4. Distance Metrics — How Far Apart?

### Three key distance measures

**Euclidean distance** (straight line):
> d(a, b) = ‖a - b‖₂ = √Σ(aᵢ - bᵢ)²

**Manhattan distance** (grid walk):
> d(a, b) = ‖a - b‖₁ = Σ|aᵢ - bᵢ|

**Cosine similarity** (angle between vectors, ignores length):
> cos_sim(a, b) = (a · b) / (‖a‖ × ‖b‖)

### When to use which?
| Metric | Best for | Avoid when |
|---|---|---|
| **Euclidean** | Continuous data, similar scales | High dimensions (curse), text |
| **Manhattan** | Grid-like data, robust to outliers | You need smooth optimization |
| **Cosine** | Text/embeddings, when length doesn't matter | When magnitude matters |

In [ ]:
# ─────────────────────────────────────────────────────────
# DISTANCE METRICS
# ─────────────────────────────────────────────────────────

# Define all three distance functions
def euclidean(a, b):
    return np.linalg.norm(a - b)        # L2 norm of the difference

def manhattan(a, b):
    return np.linalg.norm(a - b, ord=1) # L1 norm of the difference

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def cosine_distance(a, b):
    return 1 - cosine_similarity(a, b)  # 0 = same, 1 = opposite

print("=== House features: which houses are similar? ===")
print("Features: [size_sqft, bedrooms, age_years, distance_km]")
print()

houses = {
    'House A': np.array([1500., 3., 10., 5.]),
    'House B': np.array([1600., 3., 12., 4.8]),  # similar to A
    'House C': np.array([2800., 5.,  2., 1.]),   # very different
    'House D': np.array([3000., 3., 10., 5.]),   # big but otherwise similar
}

# Normalize features first (important for distance metrics!)
all_features = np.array(list(houses.values()))
means = all_features.mean(axis=0)
stds  = all_features.std(axis=0) + 1e-8  # avoid division by zero
norm_houses = {k: (v - means) / stds for k, v in houses.items()}

query = 'House A'
print(f"Distances from {query}:")
print(f"{'House':<10} {'Euclidean':>12} {'Manhattan':>12} {'Cosine sim':>12} {'Most similar by?'}")
print('-' * 60)
for name, raw_feat in houses.items():
    if name == query: continue
    a = norm_houses[query]
    b = norm_houses[name]
    print(f"{name:<10} {euclidean(a,b):>12.4f} {manhattan(a,b):>12.4f} {cosine_similarity(a,b):>12.4f}")

print()
print("House D: large size but same bedrooms/age/distance as A")
print("→ Euclidean says it's far (different size scale)")
print("→ Cosine similarity says it's similar (same proportional pattern)")
print("Which metric is 'right' depends on what you care about!")

In [ ]:
# ─────────────────────────────────────────────────────────
# APPLICATION: cosine similarity for document similarity
# (where magnitude doesn't matter, only direction)
# ─────────────────────────────────────────────────────────

print("=== Cosine similarity for text (word counts) ===")
print("Words: [python, java, machine, learning, coffee, tea]")
print()

# TF (term frequency) vectors for 5 articles
docs = {
    'Python ML tutorial':  np.array([5., 0., 4., 6., 0., 0.]),
    'Java programming':    np.array([0., 7., 0., 0., 1., 0.]),
    'Deep learning guide': np.array([2., 0., 6., 8., 0., 0.]),
    'Coffee culture':      np.array([0., 0., 0., 0., 5., 2.]),
    'Tea traditions':      np.array([0., 0., 0., 0., 1., 6.]),
}

query_doc = 'Python ML tutorial'
print(f"Finding documents similar to: '{query_doc}'")
print()
for name, vec in docs.items():
    if name == query_doc: continue
    sim = cosine_similarity(docs[query_doc], vec)
    euc = euclidean(docs[query_doc], vec)
    print(f"  '{name}':")
    print(f"    Cosine similarity: {sim:.4f}  (1=identical, 0=unrelated)")
    print(f"    Euclidean distance: {euc:.4f}")

print()
print("→ Cosine correctly identifies 'Deep learning guide' as most similar")
print("→ despite it having more words (higher magnitude)")
print("→ because cosine ignores document length, only cares about topic proportions")

---
## Summary

| Tool | Formula | NumPy | Best for |
|---|---|---|---|
| Dot product | Σ aᵢbᵢ | `np.dot(a,b)` or `a @ b` | Neuron activation, similarity |
| L1 norm | Σ\|vᵢ\| | `np.linalg.norm(v, 1)` | LASSO, sparse features |
| L2 norm | √(Σvᵢ²) | `np.linalg.norm(v)` | Ridge regularization, Euclidean |
| L∞ norm | max\|vᵢ\| | `np.linalg.norm(v, np.inf)` | Minimax |
| Frobenius | √(ΣΣAᵢⱼ²) | `np.linalg.norm(A, 'fro')` | Weight regularization |
| Euclidean dist | ‖a-b‖₂ | `np.linalg.norm(a-b)` | KNN, continuous features |
| Cosine sim | a·b / (‖a‖‖b‖) | manual formula | Text, embeddings |

**Next: Notebook 09 — Orthogonality, Projections & Gram-Schmidt**